# Short video — clip selection

Feed a long-form YouTube URL to `run_short_video_agent` and look at what it picks.

**Scope:** this notebook exercises the *decision* only — transcript in, clip
timestamps out. It does not download or cut video.

The media pipeline (yt-dlp download, ffmpeg cutting, S3 upload) moved to
`automation-tools` at `app/services/short_video/` on 2026-08-11, because that is
where the Tool owns it. It has its own tests there
(`tests/test_short_video_media.py`). Reproducing it here would be a second copy
that drifts.

Use this for prompt iteration: change `SHORTS_PROMPT` in
`ai_agents/agents/short_video_generator/tools/get_short_clips_tool.py`, re-run,
compare. Requires the `shortvideo` extra.


In [ ]:
from __future__ import annotations

import json
import time
from datetime import datetime
from pathlib import Path

import sys
sys.path.insert(0, "/Users/utkarshumang/my_projects/ai-agents-service")
sys.path.insert(0, "../..")

from dotenv import load_dotenv
load_dotenv()  # before importing ai_agents — its bootstrap checks keys at import

from ai_agents.agents.short_video_generator import (
    run_short_video_agent,
    is_valid_youtube_url,
)
from ai_agents.agents.short_video_generator.tools.get_short_clips_tool import (
    get_transcript,
)

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


## 1. Pick a video

Long-form only — the agent is looking for 60-80s windows worth lifting out.
Note the Tool caps sources at 20 minutes; nothing stops you going longer here.


In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"

assert is_valid_youtube_url(VIDEO_URL), f"rejected by the URL gate: {VIDEO_URL}"
print("ok:", VIDEO_URL)


## 2. Transcript

Pulled separately so you can see what the model is actually reading. The agent
does this internally too — captions API first, yt-dlp VTT as fallback.


In [ ]:
t0 = time.perf_counter()
transcript = get_transcript(VIDEO_URL)
print(f"{len(transcript):,} chars, ~{len(transcript)//4:,} tokens, {time.perf_counter()-t0:.1f}s")
print()
print(transcript[:800])


## 3. Run the agent

One LLM call over the whole transcript. Returns 5-10 clips of 60-80s.


In [ ]:
t0 = time.perf_counter()
analysis = run_short_video_agent(VIDEO_URL)
elapsed = time.perf_counter() - t0

print(f"{len(analysis.clips)} clips in {elapsed:.1f}s")


## 4. What it picked


In [ ]:
for i, c in enumerate(analysis.clips):
    print(f"[{i}] {c.start_timestamp} -> {c.end_timestamp}  ({c.duration_seconds}s)")
    print(f"    {c.title}")
    print(f"    topic : {c.topic}")
    print(f"    why   : {c.why_it_works}")
    print()


## 5. Sanity checks

The prompt asks for 60-80s, non-overlapping, real timestamps. It does not always
comply — worth checking before blaming the media pipeline for a bad clip.


In [ ]:
def _secs(ts: str) -> int:
    parts = [int(p) for p in ts.split(":")]
    while len(parts) < 3:
        parts.insert(0, 0)
    h, m, s = parts
    return h * 3600 + m * 60 + s

spans = sorted((_secs(c.start_timestamp), _secs(c.end_timestamp), i)
               for i, c in enumerate(analysis.clips))

off_length = [(i, e - s) for s, e, i in spans if not (60 <= e - s <= 80)]
overlaps = [(a[2], b[2]) for a, b in zip(spans, spans[1:]) if b[0] < a[1]]
inverted = [i for s, e, i in spans if e <= s]

print("outside 60-80s :", off_length or "none")
print("overlapping    :", overlaps or "none")
print("end <= start   :", inverted or "none")


## 6. Save for comparison

Keeps a record so two prompt variants can be diffed rather than eyeballed.


In [ ]:
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
out = OUTPUT_DIR / f"clips-{stamp}.json"
out.write_text(json.dumps({
    "video_url": str(analysis.video_url),
    "elapsed_s": round(elapsed, 1),
    "transcript_chars": len(transcript),
    "clips": [c.model_dump() for c in analysis.clips],
}, indent=2))
print("wrote", out)
